# Part 2: RF 模型训练及评估

请先运行 `01_dependencies_and_data.ipynb`

In [ ]:
"""Part 2: RF. Run 01_dependencies_and_data.ipynb first."""
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid, GroupKFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
import os
import dill
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json

# 控制不同评估模块的开关
RUN_GRID_SEARCH = False          # 基于 train/val 的全网格搜索（可选，默认关闭）
RUN_NESTED_CV = False           # rgiid/year 的 nested CV（可选，默认关闭）
RUN_SPATIOTEMPORAL_HOLDOUT = True  # 时空外推 holdout（主流程，默认开启）

root_dir = "C:\\ML4GM"
outputs_dir = os.path.join(root_dir, "models")

def load_pkl(filepath):
    with open(filepath, "rb") as fr:
        return dill.load(fr)

def save_pkl(filepath, data):
    with open(filepath, "wb") as fw:
        dill.dump(data, fw)
    print(f"[{filepath}] data saving...")

_data = load_pkl(os.path.join(outputs_dir, "preprocessed_data.pkl"))
# 单次划分的数据（兼容原流程）
X_train = _data["X_train"]
X_test = _data["X_test"]
y_train = _data["y_train"]
y_test = _data["y_test"]
X_val = _data.get("X_val")
y_val = _data.get("y_val")
X_scaler = _data.get("X_scaler")
y_scaler = _data.get("y_scaler")
feature_columns = _data["feature_columns"]

# y 的语义：新版本 01 会写入 y_standardized=True（train/val/test 全部标准化）
# 旧版本：y_train/y_test 标准化，但 y_val 为原始尺度
_y_standardized = _data.get("y_standardized", None)
y_train_is_std = True
if _y_standardized is True:
    y_val_is_std = True
elif _y_standardized is False:
    y_val_is_std = False
else:
    y_val_is_std = False

# 全量数据与分组（用于 nested GroupKFold / 时空 holdout）
X_all = _data.get("X_all")
y_all = _data.get("y_all")
rgiid_all = _data.get("rgiid_all")
year_all = _data.get("year_all")


### RF模型训练及评估

In [ ]:
# 基于训练集和验证集的 RF 超参数搜索（主指标：val_RMSE）

best_params = None
best_val_rmse = float("inf")

if RUN_GRID_SEARCH:
    if X_val is None or y_val is None:
        print("[GridSearch] Validation set not available, skip RF hyperparameter search.")
    else:
        param_grid = {
            "n_estimators": [100, 200, 500, 800],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2", 0.8],
            "bootstrap": [True],
            "n_jobs": [-1],
            "random_state": [42],
        }

        results = []
        grids = list(ParameterGrid(param_grid))
        print(f"[GridSearch] Total RF hyperparameter combinations: {len(grids)}")

        # 统一在原始尺度（dhdt 物理单位）上计算 val_RMSE，避免口径混乱
        if y_scaler is None:
            raise ValueError("y_scaler not found in pkl; please rerun 01 to generate y_scaler.")

        y_val_true = y_scaler.inverse_transform(y_val.reshape(-1, 1)).reshape(-1) if y_val_is_std else y_val

        for i, params in enumerate(grids, start=1):
            start_time = time.time()
            model = RandomForestRegressor(**params)
            model.fit(X_train, y_train)

            y_val_pred = model.predict(X_val)
            y_val_pred_true = y_scaler.inverse_transform(y_val_pred.reshape(-1, 1)).reshape(-1) if y_train_is_std else y_val_pred
            # 某些 sklearn 版本不支持 squared 参数，这里手动开根号得到 RMSE
            val_rmse = np.sqrt(mean_squared_error(y_val_true, y_val_pred_true))

            elapsed = time.time() - start_time
            print(f"[GridSearch][{i}/{len(grids)}] params={params}, val_RMSE={val_rmse:.4f}, time={elapsed:.1f}s")

            results.append({
                "params": json.dumps(params),
                "val_RMSE": val_rmse,
                "time_sec": elapsed,
            })

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_params = params

        print("[GridSearch] Best RF params:", best_params)
        print("[GridSearch] Best val_RMSE:", best_val_rmse)

        rf_search_df = pd.DataFrame(results)
        os.makedirs(outputs_dir, exist_ok=True)
        search_path = os.path.join(outputs_dir, "rf_optional_hparam_search.csv")
        rf_search_df.to_csv(search_path, index=False)
        print(f"[GridSearch] RF hyperparameter search results saved to {search_path}")
else:
    print("[GridSearch] Skipped (RUN_GRID_SEARCH=False). Using default/best params from other blocks.")


In [ ]:
# Nested GroupKFold evaluation for RF (supports repeated runs + year generalization)

def _safe_n_splits(requested: int, n_unique: int) -> int:
    if n_unique < 2:
        return 0
    return min(requested, n_unique)


def run_nested_groupkfold_rf(
    *,
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    group_name: str,
    outer_splits: int = 10,
    inner_splits: int = 3,
    n_iter: int = 30,
    sampler_seed: int = 42,
    run_tag: str = "run1",
):
    if X is None or y is None or groups is None:
        print(f"[{group_name}] Missing X/y/groups; skip.")
        return None

    unique_groups = np.unique(groups)
    outer_k = _safe_n_splits(outer_splits, len(unique_groups))
    if outer_k < 2:
        print(f"[{group_name}] Not enough unique groups for outer CV: {len(unique_groups)}")
        return None

    outer_cv = GroupKFold(n_splits=outer_k)

    param_distributions = {
        "n_estimators": [100, 200, 400, 800],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.8],
        "bootstrap": [True],
        "n_jobs": [-1],
        "random_state": [42],
    }

    outer_results = []

    for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"\n[{group_name}][{run_tag}] ===== Outer fold {outer_fold} / {outer_k} =====")

        X_tr_raw, X_te_raw = X[outer_train_idx], X[outer_test_idx]
        y_tr_raw, y_te_raw = y[outer_train_idx], y[outer_test_idx]
        groups_tr = groups[outer_train_idx]

        # 动态 inner 折数：保证 <= outer-train 中的 unique groups
        inner_unique = np.unique(groups_tr)
        inner_k = _safe_n_splits(inner_splits, len(inner_unique))
        if inner_k < 2:
            print(f"[{group_name}][{run_tag}] Skip outer fold {outer_fold}: not enough groups for inner CV")
            continue
        inner_cv = GroupKFold(n_splits=inner_k)

        sampled_params = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=sampler_seed))

        inner_search_logs = []
        best_inner_val_rmse = float("inf")
        best_params = None

        for i, params in enumerate(sampled_params, start=1):
            inner_rmse_list = []
            start_time = time.time()

            for inner_train_idx, inner_val_idx in inner_cv.split(X_tr_raw, y_tr_raw, groups=groups_tr):
                X_in_tr, X_in_val = X_tr_raw[inner_train_idx], X_tr_raw[inner_val_idx]
                y_in_tr, y_in_val = y_tr_raw[inner_train_idx], y_tr_raw[inner_val_idx]

                scaler = StandardScaler()
                X_in_tr_std = scaler.fit_transform(X_in_tr)
                X_in_val_std = scaler.transform(X_in_val)

                model = RandomForestRegressor(**params)
                model.fit(X_in_tr_std, y_in_tr)
                y_val_pred = model.predict(X_in_val_std)

                rmse = np.sqrt(mean_squared_error(y_in_val, y_val_pred))
                inner_rmse_list.append(rmse)

            mean_inner_rmse = float(np.mean(inner_rmse_list))
            elapsed = time.time() - start_time
            inner_search_logs.append({
                "group": group_name,
                "run": run_tag,
                "outer_fold": outer_fold,
                "trial": i,
                "params": json.dumps(params),
                "mean_inner_val_RMSE": mean_inner_rmse,
                "time_sec": elapsed,
            })
            print(
                f"[{group_name}][{run_tag}] Outer {outer_fold}, trial {i}/{n_iter}: "
                f"mean inner val_RMSE={mean_inner_rmse:.4f}, time={elapsed:.1f}s"
            )

            if mean_inner_rmse < best_inner_val_rmse:
                best_inner_val_rmse = mean_inner_rmse
                best_params = params

        print(
            f"[{group_name}][{run_tag}] Best params for outer fold {outer_fold}: "
            f"{best_params}, inner mean val_RMSE={best_inner_val_rmse:.4f}"
        )

        outer_search_df = pd.DataFrame(inner_search_logs)
        search_path = os.path.join(outputs_dir, f"rf_hparam_search_{group_name}_{run_tag}_outer{outer_fold}.csv")
        outer_search_df.to_csv(search_path, index=False)

        scaler_outer = StandardScaler()
        X_tr_std = scaler_outer.fit_transform(X_tr_raw)
        X_te_std = scaler_outer.transform(X_te_raw)

        final_model = RandomForestRegressor(**best_params)
        final_model.fit(X_tr_std, y_tr_raw)
        y_te_pred = final_model.predict(X_te_std)

        test_rmse = float(np.sqrt(mean_squared_error(y_te_raw, y_te_pred)))
        test_mae = float(mean_absolute_error(y_te_raw, y_te_pred))
        test_r2 = float(r2_score(y_te_raw, y_te_pred))
        print(
            f"[{group_name}][{run_tag}] Outer fold {outer_fold} test RMSE={test_rmse:.4f}, "
            f"MAE={test_mae:.4f}, R2={test_r2:.4f}"
        )

        outer_results.append({
            "group": group_name,
            "run": run_tag,
            "outer_fold": outer_fold,
            "outer_k": outer_k,
            "inner_k": inner_k,
            "sampler_seed": sampler_seed,
            "n_iter": n_iter,
            "test_RMSE": test_rmse,
            "test_MAE": test_mae,
            "test_R2": test_r2,
            "best_inner_val_RMSE": best_inner_val_rmse,
            "best_params": json.dumps(best_params),
        })

    outer_df = pd.DataFrame(outer_results)
    outer_path = os.path.join(outputs_dir, f"rf_nested_cv_{group_name}_{run_tag}_outer.csv")
    outer_df.to_csv(outer_path, index=False)

    if len(outer_df) > 0:
        print(f"\n[{group_name}][{run_tag}] Outer results saved to: {outer_path}")
        print("Test RMSE mean±std:", outer_df["test_RMSE"].mean(), "+-", outer_df["test_RMSE"].std())
        print("Test MAE mean±std:", outer_df["test_MAE"].mean(), "+-", outer_df["test_MAE"].std())
        print("Test R2  mean±std:", outer_df["test_R2"].mean(), "+-", outer_df["test_R2"].std())
    else:
        print(f"[{group_name}][{run_tag}] No outer results collected.")

    return outer_df


# ---- 1) Repeated nested CV by rgiid (2 runs with different sampler seeds) ----
if RUN_NESTED_CV:
    if X_all is None or y_all is None or rgiid_all is None:
        print("[NestedCV] X_all / y_all / rgiid_all not found in pkl; please rerun 01 to update preprocessed_data.pkl.")
    else:
        rgiid_runs = [("run1", 42), ("run2", 43)]
        rgiid_outer_all = []
        for run_tag, seed in rgiid_runs:
            df = run_nested_groupkfold_rf(
                X=X_all,
                y=y_all,
                groups=rgiid_all,
                group_name="rgiid",
                outer_splits=10,
                inner_splits=3,
                n_iter=30,
                sampler_seed=seed,
                run_tag=run_tag,
            )
            if df is not None and len(df) > 0:
                rgiid_outer_all.append(df)

        if len(rgiid_outer_all) > 0:
            rgiid_repeats_df = pd.concat(rgiid_outer_all, axis=0, ignore_index=True)
            repeats_path = os.path.join(outputs_dir, "rf_optional_nested_cv_rgiid_repeats_summary.csv")
            rgiid_repeats_df.to_csv(repeats_path, index=False)
            print("\n[NestedCV][rgiid] Repeats summary saved to:", repeats_path)
            print("[NestedCV][rgiid] Overall Test RMSE mean±std:", rgiid_repeats_df["test_RMSE"].mean(), "+-", rgiid_repeats_df["test_RMSE"].std())

    # ---- 2) Year-grouped nested CV for temporal generalization ----
    if X_all is None or y_all is None or year_all is None:
        print("[NestedCV] X_all / y_all / year_all not found in pkl; please rerun 01 to update preprocessed_data.pkl with year_all.")
    else:
        year_outer_df = run_nested_groupkfold_rf(
            X=X_all,
            y=y_all,
            groups=year_all,
            group_name="year",
            outer_splits=10,
            inner_splits=3,
            n_iter=30,
            sampler_seed=42,
            run_tag="run1",
        )
        if year_outer_df is not None and len(year_outer_df) > 0:
            year_path = os.path.join(outputs_dir, "rf_optional_nested_cv_year_outer.csv")
            year_outer_df.to_csv(year_path, index=False)
            print("\n[NestedCV][year] Outer results saved to:", year_path)
else:
    print("[NestedCV] Skipped (RUN_NESTED_CV=False).")


In [ ]:
# C) Spatiotemporal extrapolation: blocked year 80/20 (2000–2019) + unseen rgiid 20%
# - Train pool: train_years AND visible_rgiid
# - Final test: test_years AND unseen_rgiid

if X_all is None or y_all is None or rgiid_all is None or year_all is None:
    print("[C] Missing X_all/y_all/rgiid_all/year_all; rerun 01 to update preprocessed_data.pkl.")
else:
    # 1) blocked year 80/20
    unique_years = sorted(np.unique(year_all))
    cut = int(len(unique_years) * 0.8)
    train_years = unique_years[:cut]
    test_years = unique_years[cut:]

    # 2) unseen rgiid 20%
    unseen_ratio = 0.2
    holdout_seed = 42
    rs = np.random.RandomState(holdout_seed)
    unique_rgiid = np.unique(rgiid_all)
    rs.shuffle(unique_rgiid)
    n_unseen = max(1, int(len(unique_rgiid) * unseen_ratio))
    unseen_rgiids = set(unique_rgiid[:n_unseen])

    train_pool_mask = np.isin(year_all, train_years) & (~np.isin(rgiid_all, list(unseen_rgiids)))
    final_test_mask = np.isin(year_all, test_years) & (np.isin(rgiid_all, list(unseen_rgiids)))

    X_pool, y_pool, g_pool = X_all[train_pool_mask], y_all[train_pool_mask], rgiid_all[train_pool_mask]
    X_final, y_final = X_all[final_test_mask], y_all[final_test_mask]

    meta = {
        "train_years": ",".join(map(str, train_years)),
        "test_years": ",".join(map(str, test_years)),
        "n_unique_years": len(unique_years),
        "unseen_ratio": unseen_ratio,
        "holdout_seed": holdout_seed,
        "n_unique_rgiid_all": int(len(np.unique(rgiid_all))),
        "n_unseen_rgiid": int(len(unseen_rgiids)),
        "n_train_pool": int(X_pool.shape[0]),
        "n_final_test": int(X_final.shape[0]),
    }
    meta_path = os.path.join(outputs_dir, "rf_spatiotemporal_holdout_meta.csv")
    pd.DataFrame([meta]).to_csv(meta_path, index=False)
    print("[C] Meta saved to:", meta_path)
    print("[C] train_years:", train_years)
    print("[C] test_years:", test_years)
    print("[C] train_pool n:", X_pool.shape[0], "final_test n:", X_final.shape[0])

    if X_pool.shape[0] == 0 or X_final.shape[0] == 0:
        print("[C] Empty train_pool or final_test after split; please adjust split ratios or check data.")
    else:
        # 3) nested tuning on train pool (groups=rgiid)
        outer_unique = np.unique(g_pool)
        outer_k = _safe_n_splits(5, len(outer_unique))
        if outer_k < 2:
            print("[C] Not enough unique rgiid in train pool for outer CV:", len(outer_unique))
        else:
            outer_cv = GroupKFold(n_splits=outer_k)

            param_distributions = {
                "n_estimators": [100, 200, 400, 800],
                "max_depth": [None, 10, 20, 30],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "max_features": ["sqrt", "log2", 0.8],
                "bootstrap": [True],
                "n_jobs": [-1],
                "random_state": [42],
            }
            n_iter = 30
            sampler_seed = 42

            outer_rows = []
            best_params_across_outer = []

            for outer_fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_pool, y_pool, groups=g_pool), start=1):
                print(f"\n[C] ===== Outer fold {outer_fold}/{outer_k} (train-pool) =====")
                X_tr_raw, X_te_raw = X_pool[tr_idx], X_pool[te_idx]
                y_tr_raw, y_te_raw = y_pool[tr_idx], y_pool[te_idx]
                g_tr = g_pool[tr_idx]

                inner_unique = np.unique(g_tr)
                inner_k = _safe_n_splits(3, len(inner_unique))
                if inner_k < 2:
                    print(f"[C] Skip outer fold {outer_fold}: not enough groups for inner CV")
                    continue
                inner_cv = GroupKFold(n_splits=inner_k)

                sampled_params = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=sampler_seed))
                inner_logs = []
                best_inner = float("inf")
                best_params = None

                for trial_i, params in enumerate(sampled_params, start=1):
                    rmses = []
                    t0 = time.time()
                    for in_tr_idx, in_va_idx in inner_cv.split(X_tr_raw, y_tr_raw, groups=g_tr):
                        X_in_tr, X_in_va = X_tr_raw[in_tr_idx], X_tr_raw[in_va_idx]
                        y_in_tr, y_in_va = y_tr_raw[in_tr_idx], y_tr_raw[in_va_idx]

                        scaler = StandardScaler()
                        X_in_tr_std = scaler.fit_transform(X_in_tr)
                        X_in_va_std = scaler.transform(X_in_va)

                        m = RandomForestRegressor(**params)
                        m.fit(X_in_tr_std, y_in_tr)
                        pred = m.predict(X_in_va_std)
                        rmses.append(np.sqrt(mean_squared_error(y_in_va, pred)))

                    mean_rmse = float(np.mean(rmses))
                    inner_logs.append({
                        "outer_fold": outer_fold,
                        "trial": trial_i,
                        "params": json.dumps(params),
                        "mean_inner_val_RMSE": mean_rmse,
                        "time_sec": time.time() - t0,
                    })
                    if mean_rmse < best_inner:
                        best_inner = mean_rmse
                        best_params = params

                pd.DataFrame(inner_logs).to_csv(
                    os.path.join(outputs_dir, f"rf_spatiotemporal_holdout_inner_logs_outer{outer_fold}.csv"),
                    index=False,
                )
                best_params_across_outer.append({
                    "outer_fold": outer_fold,
                    "best_inner_val_RMSE": best_inner,
                    "best_params": json.dumps(best_params),
                })

                scaler_outer = StandardScaler()
                X_tr_std = scaler_outer.fit_transform(X_tr_raw)
                X_te_std = scaler_outer.transform(X_te_raw)

                final_model = RandomForestRegressor(**best_params)
                final_model.fit(X_tr_std, y_tr_raw)
                y_te_pred = final_model.predict(X_te_std)

                row = {
                    "outer_fold": outer_fold,
                    "outer_k": outer_k,
                    "inner_k": inner_k,
                    "n_iter": n_iter,
                    "sampler_seed": sampler_seed,
                    "test_RMSE": float(np.sqrt(mean_squared_error(y_te_raw, y_te_pred))),
                    "test_MAE": float(mean_absolute_error(y_te_raw, y_te_pred)),
                    "test_R2": float(r2_score(y_te_raw, y_te_pred)),
                    "best_inner_val_RMSE": float(best_inner),
                    "best_params": json.dumps(best_params),
                }
                outer_rows.append(row)
                print(
                    f"[C] outer {outer_fold} train-pool test RMSE={row['test_RMSE']:.4f}, "
                    f"MAE={row['test_MAE']:.4f}, R2={row['test_R2']:.4f}"
                )

            outer_df = pd.DataFrame(outer_rows)
            outer_path = os.path.join(outputs_dir, "rf_spatiotemporal_holdout_outer.csv")
            outer_df.to_csv(outer_path, index=False)
            print("\n[C] Outer (train-pool) results saved to:", outer_path)

            # 4) choose a single hyperparam set more strictly:
            #    do an independent GroupKFold random search on the full train pool (groups=rgiid)
            if len(best_params_across_outer) == 0:
                print("[C] No outer folds completed; skip final test training.")
            else:
                tune_unique = np.unique(g_pool)
                tune_k = _safe_n_splits(3, len(tune_unique))
                if tune_k < 2:
                    print("[C] Not enough unique rgiid for strict tuning on train pool; fallback to best outer-fold params.")
                    best_df = pd.DataFrame(best_params_across_outer).sort_values("best_inner_val_RMSE")
                    chosen = json.loads(best_df.iloc[0]["best_params"])
                else:
                    tune_cv = GroupKFold(n_splits=tune_k)
                    sampled_params = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=sampler_seed))
                    tune_logs = []
                    best_rmse = float("inf")
                    chosen = None

                    for trial_i, params in enumerate(sampled_params, start=1):
                        rmses = []
                        t0 = time.time()
                        for in_tr_idx, in_va_idx in tune_cv.split(X_pool, y_pool, groups=g_pool):
                            X_in_tr, X_in_va = X_pool[in_tr_idx], X_pool[in_va_idx]
                            y_in_tr, y_in_va = y_pool[in_tr_idx], y_pool[in_va_idx]

                            scaler = StandardScaler()
                            X_in_tr_std = scaler.fit_transform(X_in_tr)
                            X_in_va_std = scaler.transform(X_in_va)

                            m = RandomForestRegressor(**params)
                            m.fit(X_in_tr_std, y_in_tr)
                            pred = m.predict(X_in_va_std)
                            rmses.append(np.sqrt(mean_squared_error(y_in_va, pred)))

                        mean_rmse = float(np.mean(rmses))
                        tune_logs.append({
                            "trial": trial_i,
                            "params": json.dumps(params),
                            "mean_val_RMSE": mean_rmse,
                            "time_sec": time.time() - t0,
                        })
                        if mean_rmse < best_rmse:
                            best_rmse = mean_rmse
                            chosen = params

                    tune_path = os.path.join(outputs_dir, "rf_spatiotemporal_holdout_strict_tuning.csv")
                    pd.DataFrame(tune_logs).to_csv(tune_path, index=False)
                    print("[C] Strict tuning logs saved to:", tune_path)
                    print("[C] Chosen params (strict tuning on full train pool):", chosen)
                    meta["strict_tuning_k"] = int(tune_k)
                    meta["strict_tuning_best_rmse"] = float(best_rmse)

                # train on full train pool, evaluate on final spatiotemporal test
                scaler_full = StandardScaler()
                X_pool_std = scaler_full.fit_transform(X_pool)
                X_final_std = scaler_full.transform(X_final)

                model_full = RandomForestRegressor(**chosen)
                model_full.fit(X_pool_std, y_pool)
                y_final_pred = model_full.predict(X_final_std)

                final_metrics = {
                    "RMSE": float(np.sqrt(mean_squared_error(y_final, y_final_pred))),
                    "MAE": float(mean_absolute_error(y_final, y_final_pred)),
                    "R2": float(r2_score(y_final, y_final_pred)),
                    "chosen_params": json.dumps(chosen),
                    **meta,
                }
                final_path = os.path.join(outputs_dir, "rf_spatiotemporal_holdout_final_test.csv")
                pd.DataFrame([final_metrics]).to_csv(final_path, index=False)
                print("[C] Final spatiotemporal test metrics saved to:", final_path)
                print("[C] Final test RMSE/MAE/R2:", final_metrics["RMSE"], final_metrics["MAE"], final_metrics["R2"])

In [ ]:
# 使用最佳参数创建RF模型（若未运行搜索则使用默认参数）
if best_params is None:
    best_params = {
        "n_estimators": 94,
        "max_depth": 17,
        "max_features": 28,
        "min_samples_leaf": 1,
        "min_samples_split": 6,
        "bootstrap": False,
        "n_jobs": -1,
        "random_state": 42,
    }

print("Using RF params:", best_params)
rf_model = RandomForestRegressor(**best_params)
rf_model.fit(X_train, y_train)  # 训练模型

save_pkl(os.path.join(outputs_dir, "rf_model.pkl"), rf_model)  # 保存模型

In [ ]:
# 验证集指标（用于监控/调参；最终报告以测试集为准）
if X_val is not None and y_val is not None:
    if y_scaler is None:
        raise ValueError("y_scaler not found in pkl; please rerun 01 to generate y_scaler.")

    y_val_pred_of_rf = rf_model.predict(X_val)

    y_val_true = y_scaler.inverse_transform(y_val.reshape(-1, 1)).reshape(-1) if y_val_is_std else y_val
    y_val_pred_true = y_scaler.inverse_transform(y_val_pred_of_rf.reshape(-1, 1)).reshape(-1) if y_train_is_std else y_val_pred_of_rf

    y_val_true = y_val_true.tolist()
    y_val_pred_true = y_val_pred_true.tolist()

    print("RF Model - Validation Set:")
    print(f"MAE: {mean_absolute_error(y_val_true, y_val_pred_true)}")
    print(f"MSE: {mean_squared_error(y_val_true, y_val_pred_true)}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_val_true, y_val_pred_true))}")
    print(f"R2: {r2_score(y_val_true, y_val_pred_true)}")
else:
    print("Validation set not in pkl (run 01 with train/val/test split).")

In [ ]:
# 预测训练集
y_train_pred_of_rf = rf_model.predict(X_train)

# 反标准化
y_train_of_rf = y_scaler.inverse_transform(y_train.reshape(-1, 1)).reshape(-1).tolist()
y_train_pred_of_rf = y_scaler.inverse_transform(y_train_pred_of_rf.reshape(-1, 1)).reshape(-1).tolist()

# 计算训练集指标
print("RF Model - Training Set:")
print(f"MAE: {mean_absolute_error(y_train_of_rf, y_train_pred_of_rf)}")  # 平均绝对误差
print(f"MSE: {mean_squared_error(y_train_of_rf, y_train_pred_of_rf)}")  # 均方误差
print(f"RMSE: {np.sqrt(mean_squared_error(y_train_of_rf, y_train_pred_of_rf))}")  # 均方根误差
print(f"R2: {r2_score(y_train_of_rf, y_train_pred_of_rf)}")  # R²

In [ ]:
y_test_pred_of_rf = rf_model.predict(X_test)  # 预测测试集数值

y_test_of_rf = y_scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1).tolist()  # 反标准化
y_test_pred_of_rf = y_scaler.inverse_transform(y_test_pred_of_rf.reshape(-1, 1)).reshape(-1).tolist()  # 反标准化

print("RF Model:")
print(f"MAE: {mean_absolute_error(y_test_of_rf, y_test_pred_of_rf)}")  # 平均绝对误差
print(f"MSE: {mean_squared_error(y_test_of_rf, y_test_pred_of_rf)}")  # 均方误差
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_of_rf, y_test_pred_of_rf))}")  # 均方根误差
print(f"R2: {r2_score(y_test_of_rf, y_test_pred_of_rf)}")  # R2

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 7), dpi=100)  # 定义画布
ax.plot(y_test_of_rf, marker="", linestyle="-", linewidth=2, label="Real values")  # 画真实值
ax.plot(y_test_pred_of_rf, marker="", linestyle="-", linewidth=2, label="Predict values")  # 画预测值
ax.set_title("Comparison of real values and prediction values (RF)", fontsize=20)  # 标题
ax.set_xlabel("Data points", fontsize=14)  # x轴标签
ax.set_ylabel("Price", fontsize=14)  # y轴标签
ax.tick_params(labelsize=12)  # 设置坐标轴轴刻度大小
ax.legend(loc="best", prop={"size": 14})  # 图例
plt.show()  # 显示图像
plt.close()  # 关闭图像

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 10), dpi=100)  # 定义画布
ax.text(
    min(y_test_of_rf),
    max(y_test_of_rf),
    f"$MAE={round(mean_absolute_error(y_test_of_rf, y_test_pred_of_rf), 4)}$"
    f"\n$MSE={round(mean_squared_error(y_test_of_rf, y_test_pred_of_rf), 4)}$"
    f"\n$RMSE={round(pow(mean_squared_error(y_test_of_rf, y_test_pred_of_rf), 0.5), 4)}$"
    f"\n$R^2={round(r2_score(y_test_of_rf, y_test_pred_of_rf), 4)}$",
    verticalalignment="top",
    fontdict={"size": 14, "color": "k"},
)  # 左上角显示模型性能指标
ax.scatter(y_test_of_rf, y_test_pred_of_rf, c="none", marker="o", edgecolors="k")  # 散点图
if True:  # 是否画拟合曲线
    from sklearn.linear_model import LinearRegression
    fitting_model = LinearRegression()
    fitting_model.fit([[item] for item in y_test_of_rf], y_test_pred_of_rf)
    ax.plot(
        [min(y_test_of_rf), max(y_test_of_rf)],
        [
            fitting_model.predict([[min(y_test_of_rf)]]).item(),
            fitting_model.predict([[max(y_test_of_rf)]]).item(),
        ],
        linewidth=2,
        linestyle="--",
        color="r",
        label="Fitting curve",
    )  # 拟合曲线
ax.plot(
    [min(y_test_of_rf), max(y_test_of_rf)],
    [min(y_test_of_rf), max(y_test_of_rf)],
    linewidth=2,
    linestyle="-",
    color="r",
    label="Reference curve",
)  # 参考曲线
ax.set_title("Residual of real values and prediction values (RF)", fontsize=20)  # 标题
ax.set_xlabel("Real values", fontsize=14)  # x轴标签
ax.set_ylabel("Predict values", fontsize=14)  # y轴标签
ax.tick_params(labelsize=12)  # 设置坐标轴轴刻度大小
ax.legend(loc="lower right", prop={"size": 14})  # 图例
plt.show()  # 显示图像
plt.close()  # 关闭图像

In [ ]:
feature_columns_of_rf, feature_importances_of_rf = list(
    zip(
        *sorted(
            zip(feature_columns, rf_model.feature_importances_.tolist()),
            key=lambda x: x[1],
            reverse=True,
        ),
    )
)  # 特征重要度排序
feature_columns_of_rf = list(feature_columns_of_rf)[:20]  # 转换为列表
feature_importances_of_rf = list(feature_importances_of_rf)[:20]  # 转换为列表

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 8), dpi=100)  # 定义画布
sns.barplot(x=feature_importances_of_rf, y=feature_columns_of_rf, ax=ax)  # 画条形图
for x, y in zip(range(len(feature_columns_of_rf)), [round(item, 4) for item in feature_importances_of_rf]):  # 遍历所有特征
    ax.text(x=y, y=x, s=round(y, 4), ha="left", va="center", fontdict={"size": 12})  # 在条形图上显示数字
ax.set_title("Feature importance ranking (RF, Top 20)", fontsize=20)  # 标题
ax.set_xlabel("Importances", fontsize=14)  # x轴标签
ax.set_ylabel("Feature", fontsize=14)  # y轴标签
ax.tick_params(labelsize=12)  # 设置坐标轴轴刻度大小
plt.show()  # 显示图像
plt.close()  # 关闭图像

feature_data = pd.DataFrame({
    'Feature': feature_columns_of_rf,
    'Importance': feature_importances_of_rf
})

# 保存为CSV文件
feature_data.to_csv(os.path.join(root_dir, 'models', 'feature_importances_rf.csv'), index=False)